# Customer Radar — huấn luyện ViSoBERT v4 (mỉa mai + số sao + nhiều nguyên nhân)

**Chuẩn bị:**
1. Trên máy (sau khi đã nhập nhãn thầy và đội đã duyệt đủ trên trang Kiểm tra):
   `cd backend && node scripts/export_training_data.js`
   → `nlp_service/data/real/{train,dev,test}.jsonl` + `manifest.json`.
2. Tạo Kaggle Dataset `customer-radar-v4` chứa **thư mục `nlp_service/`** (mã nguồn + `labels.json` + `data/`),
   KHÔNG cần thư mục `checkpoints/` hay `.venv/`.
3. Notebook: GPU T4 (một GPU là đủ), Internet On, Add Data = dataset vừa tạo.

**Đầu ra** trong `/kaggle/working/`: `ckpt-v4/` (checkpoint PyTorch), `onnx-v4/` (bản ONNX int8 cho trình duyệt),
`test_report.json` (con số đo trên test mù — đây là con số được phép báo cáo).

Bộ test mù không bao giờ vào train/dev: `--exclude data/real/test.jsonl` còn loại thêm mọi câu trùng nội dung.

In [ ]:
import glob, os, shutil, subprocess, json
src = os.path.dirname(glob.glob("/kaggle/input/**/nlp_service/train.py", recursive=True)[0])
shutil.copytree(src, "/kaggle/working/nlp_service", dirs_exist_ok=True)
os.chdir("/kaggle/working/nlp_service")
print(open("data/real/manifest.json", encoding="utf-8").read())

In [ ]:
# Trọng số gốc ViSoBERT (giấy phép: nhóm đã xin phép tác giả — xem nlp_service/README.md)
from huggingface_hub import snapshot_download
base = snapshot_download("uitnlp/visobert", local_dir="/kaggle/working/visobert")
print(base)

In [ ]:
# Trộn: nhãn thật (người + thầy, có số sao) + dữ liệu chuyên ngành cũ + UIT-ViSFD (chỉ cảm xúc)
train_files = ["data/real/train.jsonl"] + [p for p in ["data/domain_train.jsonl", "data/visfd_train.jsonl"] if os.path.exists(p)]
dev = "data/real/dev.jsonl" if sum(1 for _ in open("data/real/dev.jsonl", encoding="utf-8")) >= 50 else "data/gold_dev.jsonl"
cmd = ["python", "train.py", "--train", *train_files, "--dev", dev, "--exclude", "data/real/test.jsonl",
       "--base", "/kaggle/working/visobert", "--out", "/kaggle/working/ckpt-v4",
       "--epochs", "4", "--batch-size", "32", "--max-len", "160"]
print(" ".join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
# Con số được phép báo cáo: đo trên TEST MÙ (nhãn người, chưa từng thấy khi huấn luyện)
subprocess.run(["python", "eval_real.py", "--checkpoint", "/kaggle/working/ckpt-v4",
                "--test", "data/real/test.jsonl", "--out", "/kaggle/working/test_report.json"], check=True)

In [ ]:
# Bản ONNX int8 (~98 MB) cho trình duyệt; đầu ra có thêm sarcasm_logits
subprocess.run(["pip", "install", "-q", "onnx", "onnxruntime"], check=True)
subprocess.run(["python", "deploy/export_onnx.py", "--checkpoint", "/kaggle/working/ckpt-v4", "--out", "/kaggle/working/onnx-v4"], check=True)
shutil.make_archive("/kaggle/working/ckpt-v4", "zip", "/kaggle/working/ckpt-v4")
shutil.make_archive("/kaggle/working/onnx-v4", "zip", "/kaggle/working/onnx-v4")
print(os.listdir("/kaggle/working"))